In [20]:
import pandas as pd


sales_stations = pd.read_csv('../../data/Cleared_data/sales_stations.csv', parse_dates=['date', 'created'])
stations = pd.read_csv('../../data/Cleared_data/stations.csv', parse_dates=['created'])
sales= pd.read_csv('../../data/Cleared_data/sales.csv', parse_dates=['date'])

In [21]:
sales_stations

,station_id,date,daily_sales_count,created,partner_name,employee_name,cities
0,1531,2023-07-27,2.0,2023-07-04,МТС,Галина,Курск
1,287,2023-07-29,3.0,2023-07-04,Мегафон,Галина,Московская область
2,340,2023-07-29,2.0,2023-07-04,Мегафон,Роман,Московская область
3,596,2023-07-29,1.0,2023-07-04,Мегафон,Иван,Пермь
4,586,2023-07-30,2.0,2023-07-04,Мегафон,Иван,Екатеринбург
...,...,...,...,...,...,...,...
505371,3918,2026-09-16,1.0,2026-05-22,Beeline,Кирилл,Малый город
505372,3930,2026-09-16,1.0,2026-05-22,Beeline,Кирилл,Томск
505373,3974,2026-09-16,1.0,2026-08-14,Inventive Group,Екатерина,Омск
505374,4001,2026-09-16,2.0,2026-08-25,Inventive Group,Екатерина,Санкт-Петербург


In [22]:
sales_stations.info()

<class 'pandas.DataFrame'>
RangeIndex: 505376 entries, 0 to 505375
Data columns (total 7 columns):
 #   Column             Non-Null Count   Dtype         
---  ------             --------------   -----         
 0   station_id         505376 non-null  int64         
 1   date               505376 non-null  datetime64[us]
 2   daily_sales_count  505376 non-null  float64       
 3   created            505376 non-null  datetime64[us]
 4   partner_name       505376 non-null  str           
 5   employee_name      505376 non-null  str           
 6   cities             505376 non-null  str           
dtypes: datetime64[us](2), float64(1), int64(1), str(3)
memory usage: 27.0 MB


In [23]:
def full_station(station_id):
    stations_station = stations[stations['id'] == station_id].iloc[0]
    created = stations_station['created']
    partner_name = stations_station['partner_name']
    employee_name = stations_station['employee_name']
    city = stations_station['cities']

    sales_stations_station = sales_stations[sales_stations['station_id'] == station_id]
    last_sale = sales_stations_station['date'].max()

    date_range = pd.date_range(start=created, end=last_sale, freq='D').date

    # приводим ключи словаря к date, а не Timestamp
    sales_by_date = (
        sales_stations_station
        .assign(date=sales_stations_station['date'].dt.date)
        .set_index('date')['daily_sales_count']
        .to_dict()
    )

    rows = []
    for date_ in date_range:
        rows.append({
            'station_id': station_id,
            'date': date_,
            'daily_sales_count': sales_by_date.get(date_, 0),
            'created': created,
            'partner_name': partner_name,
            'employee_name': employee_name,
            'cities': city
        })

    answer = pd.DataFrame(rows)
    return answer


In [24]:
sparse_sales_stations = pd.concat(
    [full_station(sid) for sid in stations['id'].unique()],
    ignore_index=True
)

In [25]:
sparse_sales_stations.sort_values(by='date')

,station_id,date,daily_sales_count,created,partner_name,employee_name,cities
0,277,2023-07-04,0.0,2023-07-04,Мегафон,Екатерина,Санкт-Петербург
219256,1531,2023-07-04,0.0,2023-07-04,МТС,Галина,Курск
102185,416,2023-07-04,0.0,2023-07-04,Мегафон,Екатерина,Владивосток
219033,1530,2023-07-04,0.0,2023-07-04,МТС,Галина,Малый город
218808,1529,2023-07-04,0.0,2023-07-04,МТС,Галина,Тверь
...,...,...,...,...,...,...,...
577199,2285,2026-09-16,1.0,2023-12-11,Beeline,Екатерина,Малый город
820086,2602,2026-09-16,1.0,2024-03-13,Мегафон,Галина,Тула
1232165,3158,2026-09-16,1.0,2024-04-01,Мегафон,Кирилл,Барнаул
869207,2666,2026-09-16,1.0,2024-03-19,Мегафон,Иван,Малый город


In [26]:
sparse_sales_stations.info()

<class 'pandas.DataFrame'>
RangeIndex: 1483922 entries, 0 to 1483921
Data columns (total 7 columns):
 #   Column             Non-Null Count    Dtype         
---  ------             --------------    -----         
 0   station_id         1483922 non-null  int64         
 1   date               1483922 non-null  object        
 2   daily_sales_count  1483922 non-null  float64       
 3   created            1483922 non-null  datetime64[us]
 4   partner_name       1483922 non-null  str           
 5   employee_name      1483922 non-null  str           
 6   cities             1483922 non-null  str           
dtypes: datetime64[us](1), float64(1), int64(1), object(1), str(3)
memory usage: 79.3+ MB


In [32]:
train_sparse = sparse_sales_stations[pd.to_datetime(sparse_sales_stations['date']) <= pd.to_datetime('2025-07-31')]
test_sparse = sparse_sales_stations[pd.to_datetime(sparse_sales_stations['date']) > pd.to_datetime('2025-07-31')]

In [39]:
train_sparse.to_csv('../../data/Cleared_data/train_sparse.csv', index=False)
test_sparse.to_csv('../../data/Cleared_data/test_sparse.csv', index=False)